In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 14, Solving textbook problems with Problem wrappers

Companion notebook to [14_problem_wrappers.md](14_problem_wrappers.md). The wrapper layer (`SymplecticProblem`, `KoszulProblem`, `BianchiProblem`, ...) bundles `(structure, designated operands, registry, engine)` so you don't re-wire the same axioms for every related question.

## Setting up a `SymplecticProblem`

Three inputs: the symplectic form `ω`, the functions you want as Hamiltonians, and a registry. Everything else is auto-wired, `Closed(ω)` and `NonDegenerate(ω)` are declared on the registry, `X_f`/`X_g` are built with their defining relation `ι_{X_f} ω = ±df` registered on the engine.

In [ ]:
from jacopy.core.expr import Symbol
from jacopy.core.properties import Graded, Scalar
from jacopy.core.registry import PropertyRegistry
from jacopy.library.symplectic_problem import SymplecticProblem

reg = PropertyRegistry()
omega = Symbol("ω"); reg.declare(omega, Graded(degree=2))
f = Symbol("f"); reg.declare(f, Scalar())
g = Symbol("g"); reg.declare(g, Scalar())

prob = SymplecticProblem(omega, [f, g], registry=reg)
print(prob)

## Question 2a, Hamiltonian invariance

The canonical first question on a symplectic form: `L_{X_f} ω = 0`. Cartan magic + closed form + d² = 0 close this in eight named steps.

In [ ]:
chain = prob.prove_hamiltonian_invariance(f)
print(f"closure: {chain.initial} → {chain.final}")
print(f"steps  : {len(chain)}")
for i, step in enumerate(chain.steps):
    print(f"  [{i:02d}] {step.rule:35s} {step.before} → {step.after}")

Every step has a named rule, `L_X := d∘ι_X + ι_X∘d`, `Closed: d(ω) = 0`, `d² = 0`, ..., so the chain reads end-to-end as the textbook computation.

## Question 2c, Hamiltonian equality

Two helpers close the family: `prove_vector_field_equality` (reduces `Y = Z` via non-degeneracy) and `prove_hamiltonian_equality` (closes `ι_Y ω = ±dh`). The simplest demonstration is the reflexive case, each Hamiltonian equals itself.

In [ ]:
Xf = prob.hamiltonian(f)
chain = prob.prove_vector_field_equality(Xf, Xf)
print(f"steps: {len(chain)}, final: {chain.final}")

Three steps: the obstruction `ι_{X_f} ω − ι_{X_f} ω` peels through the `NonDegenerate` rule to `X_f − X_f`, which `simplify` collapses to `0`. The non-trivial use is when `Y` is a Lie bracket of two Hamiltonians and you want it recognised as a third Hamiltonian, same call, different operands.

## Setting up a `KoszulProblem`

Form-side counterpart: a Poisson bivector `π` with a list of forms. The wrapper auto-declares `Antisymmetric(π)` and exposes the Koszul bracket plus the tilde calculus rules (L̃ / ι̃ / d̃) on its engine.

In [ ]:
from jacopy.library.koszul_problem import KoszulProblem

reg2 = PropertyRegistry()
pi    = Symbol("π"); reg2.declare(pi,    Graded(degree=2))
alpha = Symbol("α"); reg2.declare(alpha, Graded(degree=1))
beta  = Symbol("β"); reg2.declare(beta,  Graded(degree=1))

kprob = KoszulProblem(pi, [alpha, beta], registry=reg2)
print(kprob)
print(f"bracket: {kprob.koszul_bracket}")
print(f"sharp  : {kprob.sharp}")

The bracket-expansion rule is exposed directly, useful when you want to reduce a `[α,β]_K` expression by hand without running a full proof closure.

In [ ]:
from jacopy.brackets.base import BracketApply

raw = BracketApply(kprob.koszul_bracket, alpha, beta)
print("input :", raw)
print("output:", kprob.bracket_expansion_rule.rewrite(raw))

The rewrite recovers the classical Koszul formula
`L_{π^♯α} β − L_{π^♯β} α − d⟨π^♯α, β⟩` term by term.

## When to step outside the wrapper

The wrapper is a convenience, not a wall.

1. **`prob.engine` is the pre-wired engine**, feed it to `prove_equivalence(..., engine=prob.engine)` to drive any equality under the wrapper's axiom set.
2. **`prob.hamiltonian(f)` returns the registered Hamiltonian**, build expressions with it directly.
3. **The wrapper never overrides existing declarations**, pre-declaring is the way to opt out of a default convention.

In [ ]:
# Demo: drive an arbitrary equality through prob.engine.
from jacopy.algebra.derivation import Act
from jacopy.calculus.exterior_d import d
from jacopy.core.expr import Integer
from jacopy.proof import prove_equivalence

# d(d(f)) = 0, the engine knows d² = 0 because the wrapper
# inherited it from default_engine.
chain = prove_equivalence(
    Act(d, Act(d, f)), Integer(0),
    registry=prob._registry, engine=prob._engine,
)
print(f"closure: {chain.initial} → {chain.final} ({len(chain)} steps)")

## Anatomy of the wrapper family

| Wrapper | Bundles |
|---|---|
| `SymplecticProblem` | `(M, ω, π?, {f_i}, registry, engine)` |
| `KoszulProblem` | `(π, ρ = π^♯, K, {α_i})` + tilde calculus |
| `BianchiProblem` | `(connection, registry)`, T̃-Bianchi I/II |
| `CartanFormPropertyProblem` | `(connection, frame)`, §3.1.6 |
| `CartanStructureProblem` | `(connection, frame)`, Cartan I/II |
| `KoszulConnectionProblem` | facade over the three above |

Each `__init__` validates inputs + declares structural axioms; properties expose the underlying objects; `prove_*` methods drive closures via a pre-built engine.

## Summary

* Problem wrappers bundle `(structure, designated operands, registry, engine)` so you don't re-wire axioms every question.
* `SymplecticProblem` covers form-side problems, Hamiltonian invariance, vector-field equality, Hamiltonian equality.
* `KoszulProblem` covers Poisson form-bracket problems, `[α, β]_K` expansion, tilde calculus, derivator engines.
* The pre-built engine (`prob.engine`) is exposed for closures outside the named helpers.
* Wrappers never override existing registry declarations, pre-declaring is the override mechanism.